# Do senior-high learners choose their strand — or take what their school runs?

**Source:** DepEd Learner Information System, school-level enrollment, SY 2023–2024 (as of 31 January 2024).
**Grain:** one row per school; enrollment is one row per school × grade × strand × gender.

This notebook is the working behind the writeup in `README.md`. Every number there comes from a cell here.

**The assumption this whole analysis rests on:** the raw file has no "strands offered" column. `Modified COC` tells you which *levels* a school runs (elementary, junior high, senior high) but not which *strands*. So strand availability is inferred from enrollment: a school "runs" a strand if at least one learner is enrolled in it. A strand offered on paper with no enrollees looks the same as one not offered at all. Cell 4 measures how big that assumption is.

In [1]:
import pandas as pd
pd.set_option("display.width", 120)

schools = pd.read_parquet("../../data/schools.parquet")
enrol   = pd.read_parquet("../../data/enrollment.parquet")

print(f"schools    {len(schools):,} rows")
print(f"enrollment {len(enrol):,} rows")

schools    60,167 rows
enrollment 3,489,686 rows


## 1. Build the senior-high slice

Grades 11 and 12 only, enrollment above zero, strand recorded. The raw file mixes two separators (`ACAD - ABM` but `ACAD STEM`), so labels are normalised here rather than in every chart.

In [2]:
STRAND_LABEL = {
    "ACAD - ABM": "ABM", "ACAD - HUMSS": "HUMSS", "ACAD GAS": "GAS",
    "ACAD PBM": "PBM", "ACAD STEM": "STEM",
    "TVL": "TVL", "ARTS": "Arts & Design", "SPORTS": "Sports",
}

shs = (enrol[enrol.grade.isin(["G11","G12"]) & (enrol.enrollment > 0) & enrol.strand.notna()]
       .merge(schools[["BEIS School ID","Region","Sector","Modified COC"]],
              on="BEIS School ID", how="left"))
shs["strand_label"] = shs.strand.map(STRAND_LABEL)

national = int(shs.enrollment.sum())
active   = shs["BEIS School ID"].nunique()
print(f"senior-high learners  {national:,}")
print(f"schools running SHS   {active:,}")

senior-high learners  4,130,076
schools running SHS   12,571


## 2. Validation — before believing any of it

A join that silently drops rows changes every number downstream. These run before the analysis, not after.

In [3]:
assert shs.strand_label.notna().all(), "unmapped strand label"
assert shs.Region.notna().all(), "null region after join"
assert int(shs.groupby("Region").enrollment.sum().sum()) == national
assert int(shs.groupby("strand_label").enrollment.sum().sum()) == national
print("Regional and strand totals both reconcile to the national figure.")

Regional and strand totals both reconcile to the national figure.


## 3. How big is the inference assumption?

Compare schools DepEd *classifies* as senior-high-offering against schools that actually have senior-high enrollment. The difference is the blind spot.

In [4]:
classified = schools[schools["Modified COC"].isin(
    {"JHS with SHS","All Offering","Purely SHS"})]["BEIS School ID"].nunique()

print(f"classified as SHS-offering  {classified:,}")
print(f"with SHS enrollment > 0     {active:,}")
print(f"invisible to this analysis  {classified - active:,}  "
      f"({(classified-active)/classified*100:.1f}% of SHS schools)")

classified as SHS-offering  12,793
with SHS enrollment > 0     12,571
invisible to this analysis  222  (1.7% of SHS schools)


**Read this carefully.** 222 schools are classified as senior-high-offering but report no senior-high enrollment. They are excluded here. That is 1.7% of senior high schools — small enough not to move the regional picture, but it means every availability figure below is a *slight undercount*. Stated, not hidden.

## 4. Supply — how many strands does each school actually run?

In [5]:
per_school = (shs.groupby(["Region","BEIS School ID"])["strand_label"].nunique()
                 .rename("strands").reset_index()
                 .merge(shs.groupby("BEIS School ID").enrollment.sum().rename("learners"),
                        on="BEIS School ID"))

assert per_school.strands.between(1, 8).all()
assert int(per_school.learners.sum()) == national

print(per_school.strands.value_counts().sort_index().rename("schools").to_frame())
print(f"\nAverage strands per school: {per_school.strands.mean():.2f} of 8")

         schools
strands         
1           3588
2           3294
3           2529
4           2051
5            960
6            123
7             26

Average strands per school: 2.52 of 8


## 5. The correction that changes the story

The share of *schools* running one strand and the share of *learners* in them are very different numbers. Leading with the first would have overstated the problem.

In [6]:
pct_schools  = (per_school.strands == 1).mean() * 100
pct_learners = per_school.loc[per_school.strands == 1, "learners"].sum() / national * 100

print(f"schools running only one strand   {(per_school.strands==1).sum():,}  ({pct_schools:.1f}%)")
print(f"learners in those schools         {pct_learners:.1f}%")
print(f"\nSingle-strand schools average "
      f"{per_school.loc[per_school.strands==1,'learners'].mean():.0f} learners, "
      f"against {per_school.loc[per_school.strands>1,'learners'].mean():.0f} elsewhere.")

schools running only one strand   3,588  (28.5%)
learners in those schools         8.6%

Single-strand schools average 98 learners, against 420 elsewhere.


Single-strand schools are small. Nationally they hold under a tenth of senior-high learners, so *"learners have no choice"* is not a claim this data supports. The regional breakdown is where the real variation sits.

## 6. By region

In [7]:
region = per_school.groupby("Region").agg(
    schools=("BEIS School ID","nunique"),
    avg_strands=("strands","mean"),
    learners=("learners","sum"))
region["pct_schools_one_strand"]  = (per_school[per_school.strands==1]
    .groupby("Region").size() / region.schools * 100).fillna(0)
region["pct_learners_one_strand"] = (per_school[per_school.strands==1]
    .groupby("Region").learners.sum() / region.learners * 100).fillna(0)

assert int(region.schools.sum()) == active
region.round(2).sort_values("pct_learners_one_strand", ascending=False)

,schools,avg_strands,learners,pct_schools_one_strand,pct_learners_one_strand
Region,,,,,
BARMM,344,2.00,95311,39.24,22.50
CARAGA,485,1.97,108913,52.16,19.56
CAR,358,1.99,70585,44.97,16.16
Region VIII,576,2.23,180954,35.59,14.34
Region V,977,2.46,266326,31.01,13.28
Region XI,628,2.36,214749,33.60,11.74
Region VII,1168,2.35,339174,33.39,11.10
Region VI,938,2.30,331315,33.05,9.67
MIMAROPA,430,2.53,125067,26.28,9.25


### Hand-check

Recompute one region a different way. If it does not match, something upstream is wrong.

In [8]:
caraga = shs[shs.Region == "CARAGA"].groupby("BEIS School ID").strand_label.nunique()
print(f"CARAGA recomputed: {caraga.mean():.2f}   table: {region.loc['CARAGA','avg_strands']:.2f}")
assert round(caraga.mean(), 2) == round(region.loc["CARAGA","avg_strands"], 2)

CARAGA recomputed: 1.97   table: 1.97


## 7. Availability against enrollment share, per strand

In [9]:
strand_tbl = pd.DataFrame({
    "pct_schools_running": (shs.groupby("strand_label")["BEIS School ID"].nunique()/active*100).round(1),
    "pct_of_enrollment":   (shs.groupby("strand_label").enrollment.sum()/national*100).round(1),
    "avg_learners_per_running_school": (shs.groupby("strand_label").enrollment.sum()
        / shs.groupby("strand_label")["BEIS School ID"].nunique()).round(0),
}).sort_values("pct_of_enrollment", ascending=False)

assert abs(strand_tbl.pct_of_enrollment.sum() - 100) < 0.5
strand_tbl

,pct_schools_running,pct_of_enrollment,avg_learners_per_running_school
strand_label,,,
TVL,59.2,29.5,164.0
HUMSS,58.1,26.3,149.0
STEM,37.8,18.3,159.0
GAS,49.8,14.8,98.0
ABM,43.6,10.4,79.0
Arts & Design,2.2,0.4,59.0
Sports,1.0,0.2,59.0
PBM,0.5,0.1,89.0


The two columns are not directly comparable — a school runs several strands, so the availability column sums past 100% while enrollment share sums to 100. Read them side by side, not as a subtraction.

## 8. STEM by sector

STEM needs laboratories and specialist teachers, so it is the strand most likely to track school resources rather than learner interest.

In [10]:
sector = pd.DataFrame({
    "shs_schools":   shs.groupby("Sector")["BEIS School ID"].nunique(),
    "running_stem":  shs[shs.strand_label=="STEM"].groupby("Sector")["BEIS School ID"].nunique(),
})
sector["pct_running_stem"] = (sector.running_stem/sector.shs_schools*100).round(1)
sector.sort_values("pct_running_stem", ascending=False)

,shs_schools,running_stem,pct_running_stem
Sector,,,
PSO,20,18,90.0
SUCsLUCs,139,99,71.2
Private,4725,2676,56.6
Public,7687,1953,25.4


## 9. Charts

Built by `analysis.py`, which is the reproducible version of this notebook. Run it from the repo root:

```bash
python analysis/strand-availability/analysis.py
```

Chart rules applied: axis starts at zero, units named on the axis, a national reference line so every bar has a benchmark, and titles that state the finding instead of naming the topic.

## 10. What this cannot tell you

- **Availability is inferred, not stated.** No offerings column exists; a strand counts as running only if someone enrolled in it. 222 classified senior high schools report zero enrollment and are excluded.
- **This cannot separate preference from availability.** It shows they are entangled. Settling it needs data this file does not contain — what learners applied for against what they enrolled in.
- **One school year, one snapshot.** No trend, no cohort followed over time.
- **The grain is the school, not the learner.** Nothing here follows an individual student, and nothing explains why anyone enrolled where they did.
- **Region is where the learner's school is,** not where the learner lives. A learner crossing a regional boundary to reach a school with more strands appears here as if they had that choice locally.